# 6.7. GPUs
D2L의 GPUs장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [12]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader, TensorDataset 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. GPU 사용하기

딥러닝에서는 행렬 연산을 매우 많이 수행한다. CPU도 행렬 연산을 할 수 있지만, GPU는 많은 계산을 동시에 처리하는 병렬 연산에 특화되어 있다.

그래서 신경망처럼 대규모 행렬 연산을 반복하는 작업에는 GPU를 사용하면 학습 속도를 크게 높일 수 있다. PyTorch에서는 연산이 어디에서 수행되는지를 `device`라는 개념으로 관리한다.

대표적으로 두 가지가 있다.

- `cpu` : CPU에서 계산
- `cuda` : NVIDIA GPU에서 계산

## 2. GPU 사용 가능 여부 확인

In [ ]:
import torch
from torch import nn

print(torch.cuda.is_available()) 
# GPU가 현재는 없다. 사용할 수 있다면 True가 나온다

False


`torch.cuda.is_available()`은 현재 PyTorch가 CUDA GPU를 사용할 수 있는지 확인한다. GPU가 컴퓨터에 장착되어 있더라도 CUDA를 지원하는 PyTorch가 설치되어 있지 않으면 False가 나올 수 있다.

터미널에 GPU 자체를 명령으로 확인할 수도 있다.

    nvidia-smi

## 3. Device 설정

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device) # GPU 쓸수 있으면 cuda 없으면 cpu가 뜬다.

cpu


보통 PyTorch 코드에서는 먼저 사용할 device를 하나 정한다.

```py
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 이렇게 하면 GPU가 있으면 GPU쓰고 없으면 CPU를 쓴다.
```

그래서 같은 코드를 GPU PC, CPU PC 모두에서 실행할 수 있다.

## 4. Tensor는 기본적으로 CPU에 만들어진다.

In [3]:
x = torch.tensor([1, 2, 3])

print(x)
print(x.device)

tensor([1, 2, 3])
cpu


PyTorch Tensor는 device를 따로 지정하지 않으면 기본적으로 CPU 메모리에 만들어진다.

Tensor가 어느 device에 존재하는지는 `.device`로 확인할 수 있다.

## 5. Tensor를 GPU로 이동시키기

In [4]:
x = torch.tensor([1, 2, 3])

x = x.to(device)

print(x)
print(x.device)

tensor([1, 2, 3])
cpu


Tensor를 GPU에서 계산하려면 Tensor 자체를 GPU 메모리로 이동시켜야 한다.

```python
x = x.to(device)
```

`.to(device)`는 Tensor를 지정한 device로 이동시킨다.

예를 들어서
```text
CPU RAM
   ↓
x.to("cuda")
   ↓
GPU VRAM
```

이렇게 만들 때부터 GPU에 만드는 것도 가능하다.
```py
x = torch.ones(2, 3, device=device)

print(x)
print(x.device)
```

## 6. 연산하는 Tensor들은 같은 device에 있어야 한다.

In [5]:
x = torch.ones(3).to(device)
y = torch.ones(3).to(device)

z = x + y

print(z)
print(z.device)

tensor([2., 2., 2.])
cpu


같이 연산하는 Tensor들은 반드시 같은 device에 있어야 한다.

예를 들어

```text
x -> GPU
y -> GPU

x + y -> 가능

x -> GPU
y -> CPU

x + y -> 오류

CPU메모리와 GPU 메모리는 서로 다른 공간이라 그렇다.
```

In [ ]:
x = torch.ones(3).to(device)
y = torch.ones(3)

print(x.device)
print(y.device)

# x + y

## 7. GPU가 여러개라면?

만약 GPU가 여러 개 존재한다면 각 GPU에는 번호가 붙는다.

```text
cuda:0 → 첫 번째 GPU
cuda:1 → 두 번째 GPU
cuda:2 → 세 번째 GPU
```

In [ ]:
print(torch.cuda.device_count()) # 개수 확인

0


## 8. 신경망 모델도 GPU로 옮겨야 한다.

Tensor만 옮기면 안된다.

In [7]:
model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

model = model.to(device)

model

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)

신경망의 Weight와 Bias도 Tensor이기 때문에 GPU에서 학습하려면 모델 또한 GPU로 이동시켜야 한다.

In [8]:
print(model[0].weight.device)
print(model[0].bias.device)

cpu
cpu


## 9. 입력 데이터와 모델이 같은 GPU에 있어야 한다.

실제 학습에서 실수가 많이 나오는 곳이라고 한다.

In [9]:
X = torch.randn(32, 4).to(device)

model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
).to(device)

y_hat = model(X)

print(X.device)
print(next(model.parameters()).device)
print(y_hat.device)

cpu
cpu
cpu


GPU에서 신경망을 실행하려면

- 입력 X
- 모델 Weight
- 모델 Bias

가 모두 같은 GPU에 있어야 한다.

예를 들어

```text
X       → cuda:0
Weight  → cuda:0
Bias    → cuda:0

        ↓

Forward 연산

        ↓

y_hat   → cuda:0
```

## 10. 실제 학습에선 어떻게 사용하나

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
).to(device)

X_train = torch.randn(64, 4)
y_train = torch.randn(64, 1)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=16)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for X, y in train_loader:

    X = X.to(device)
    y = y.to(device)

    y_hat = model(X)

    loss = loss_fn(y_hat, y)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

GPU 학습의 기본 흐름은 다음과 같다.

```text
1. device 결정

device = cuda 또는 cpu

2. 모델 이동

model.to(device)

3. 배치 데이터 이동

X.to(device)
y.to(device)

4. 순전파

y_hat = model(X)

5. Loss 계산

loss = loss_fn(y_hat, y)

6. 역전파

loss.backward()

7. Weight / Bias 업데이트

optimizer.step()
```

이건 CNN, RNN, Transformer를 학습하더라도 거의 그대로 유지되는 패턴이다.

## 11. CPU - GPU 데이터 이동은 공짜가 아니다

GPU가 빠르다고 해서 무조건 모든 작업이 빨라지는 것은 아니다. CPU와 GPU는 서로 다른 메모리를 사용하기 때문에 데이터를 이동시키는 데 시간이 필요하다.

```text
CPU RAM
   ↓ 데이터 전송
GPU VRAM
   ↓ 계산
GPU VRAM
   ↓ 데이터 전송
CPU RAM
```

그래서 CPU, GPU 사이에서 데이터를 계속 왔다 갔다 하면 성능이 오히려 떨어질 수 있다. 가능하면 Tensor와 모델을 GPU에 올려둔 상태에서 여러 연산을 연속해서 수행하는 것이 좋다.

D2L에선 GPU의 계산은 빠르지만 device 사이의 데이터 전송은 상대적으로 느리므로 불필요한 복사를 피해야 한다고 강조한다.

NumPy는 CPU에서 동작하므로 GPU Tensor를 NumPy로 바꾸려면

In [14]:
x_numpy = x.cpu().numpy() # CPU로 가져와야한다.

## 12. 오늘의 정리

- 딥러닝에서는 많은 행렬 연산을 수행하기 때문에 GPU를 사용하면 학습 속도를 높일 수 있다.
- PyTorch에서는 CPU와 GPU를 `device`로 구분한다.
- Tensor는 기본적으로 CPU에 생성된다.
- Tensor를 GPU로 보내려면 `.to(device)`를 사용한다.
- 신경망 모델도 `.to(device)`를 사용해 GPU로 이동시켜야 한다.
- 모델과 입력 데이터는 반드시 같은 device에 있어야 한다.
- GPU가 여러 개라면 `cuda:0`, `cuda:1`처럼 구분한다.
- GPU의 Weight와 Bias 역시 Tensor이므로 모델을 GPU로 옮기면 Weight와 Bias도 함께 이동한다.
- CPU와 GPU 사이의 데이터 전송에는 비용이 발생하므로 불필요하게 데이터를 계속 이동시키면 안 된다.